In [1]:
import torch
from model import Model
from train import train
from graph_generation import random_graph
import algorithms

#### Experimental setup

In [2]:
num_nodes = 6 # we will consider weighted undirected graph of 6 nodes in the example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Create the model

In [3]:
model = Model(['bfs', 'bf', 'prim', 'cc'], in_dim=1, hidden_dim=32, out_dim=1).to(device)

#### Train the model

In [4]:
train(model, train_size=100, val_size=20, num_nodes=6, epochs=20, patience=20)

Generating training data... Generated!
Start training process...
Epoch 1 | train loss = 0.79 (state=0.47, predec=0.19, term=1.13) | val loss = 0.51 (state=0.24, predec=0.17, term=0.94)
Epoch 2 | train loss = 0.42 (state=0.18, predec=0.13, term=0.89) | val loss = 0.39 (state=0.17, predec=0.12, term=0.79)
Epoch 3 | train loss = 0.36 (state=0.14, predec=0.13, term=0.81) | val loss = 0.37 (state=0.15, predec=0.12, term=0.81)
Epoch 4 | train loss = 0.32 (state=0.11, predec=0.12, term=0.78) | val loss = 0.29 (state=0.10, predec=0.10, term=0.72)
Epoch 5 | train loss = 0.29 (state=0.09, predec=0.11, term=0.75) | val loss = 0.26 (state=0.07, predec=0.10, term=0.70)
Epoch 6 | train loss = 0.27 (state=0.08, predec=0.10, term=0.74) | val loss = 0.31 (state=0.11, predec=0.10, term=0.78)
Epoch 7 | train loss = 0.25 (state=0.07, predec=0.09, term=0.71) | val loss = 0.26 (state=0.09, predec=0.08, term=0.68)
Epoch 8 | train loss = 0.24 (state=0.06, predec=0.08, term=0.71) | val loss = 0.22 (state=0.06,

#### Simulate Breadth-First-Search (BFS)

In [5]:
# generate graph
graph, source = random_graph(num_nodes=num_nodes, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 0

# generate bfs states
states, predecessors, inf, terminations = algorithms.compute_states('bfs', graph, source)
steps = algorithms.generate_steps(states, predecessors, terminations)

# input of the model prediction
edges = graph.get_edge_tensors(device)
h = torch.zeros(graph.num_nodes, model.hidden_dim, device=device)
x = torch.tensor(states[0], dtype=torch.float32).unsqueeze(1).to(device)

for step, (state, next_state, predecessors, termination) in enumerate(steps):
    state_pred, p_pred, h, term_pred = model('bfs', edges, x, h)

    # convert the state prediction into probabilities
    prediction = [1 if p > 0.5 else 0 for p in torch.sigmoid(state_pred).squeeze(1)]

    print(f"Step {step + 1}")
    print(f"Prediction: {prediction}")
    print(f"Target:     {next_state}")

    # termination prediction
    if torch.sigmoid(term_pred).item() > 0.5:
        break

    h = h.detach()
    x = torch.sigmoid(state_pred).detach() # output of previous iteration becomes input of the next

Step 1
Prediction: [1, 1, 1, 0, 0, 1]
Target:     [1, 1, 1, 0, 0, 1]


#### Simulate Bellman-Ford (BF)

In [6]:
# generate graph
graph, source = random_graph(num_nodes=num_nodes, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 0

# generate bf states
states, predecessors, inf, terminations = algorithms.compute_states('bf', graph, source)
steps = algorithms.generate_steps(states, predecessors, terminations)

# input of the model prediction
edges = graph.get_edge_tensors(device)
h = torch.zeros(graph.num_nodes, model.hidden_dim, device=device)
x = torch.tensor(states[0], dtype=torch.float32).unsqueeze(1).to(device)

for step, (state, next_state, predecessors, termination) in enumerate(steps):
    state_pred, p_pred, h, term_pred = model('bf', edges, x, h)

    prediction = [round(x.item(), 2) for x in state_pred.squeeze(1)]
    target = [round(x, 2) for x in next_state]

    print(f"Step {step + 1}")
    print(f"Prediction: {prediction}")
    print(f"Target:     {target}")

    # termination prediction
    if torch.sigmoid(term_pred).item() > 0.5:
        break

    h = h.detach()
    x = torch.sigmoid(state_pred).detach() # output of previous iteration becomes input of the next

# final shortest path comparison
true_shortest = [round(x, 2) for x in states[-1]]
pred_shortest = [round(x.item(), 2) for x in state_pred.squeeze(1)]
print(f"\nShortest path prediction vs true:")
print(f"Prediction: {pred_shortest}")
print(f"Target:     {true_shortest}")

Step 1
Prediction: [0.17, 1.49, 3.15, 3.43, 3.15, 3.02]
Target:     [0.0, 1.95, 2.95, 2.95, 2.95, 2.95]

Shortest path prediction vs true:
Prediction: [0.17, 1.49, 3.15, 3.43, 3.15, 3.02]
Target:     [0.0, 1.95, 2.95, 2.95, 2.95, 2.95]


In [7]:
# generate graph
graph, source = random_graph(num_nodes=6, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 0

# generate bf states
states, predecessors_by_step, inf, terminations = algorithms.compute_states('bf', graph, source)
steps = algorithms.generate_steps(states, predecessors_by_step, terminations)

# input of the model prediction
edges = graph.get_edge_tensors(device)
h = torch.zeros(graph.num_nodes, model.hidden_dim, device=device)
x = torch.tensor(states[0], dtype=torch.float32).unsqueeze(1).to(device)

for step, (state, next_state, step_predecessors, termination) in enumerate(steps):
    state_pred, p_pred, h, term_pred = model('bf', edges, x, h)

    prediction = [round(v.item(), 2) for v in state_pred.squeeze(1)]
    target = [round(v, 2) for v in next_state]

    print(f"Step {step + 1}")
    print(f"Prediction: {prediction}")
    print(f"Target:     {target}")

    if torch.sigmoid(term_pred).item() > 0.5:
        break

    h = h.detach()
    x = state_pred.detach()

# path reconstruction
def reconstruct_path(node, pred_array, source):
    path, visited = [node], set()
    while path[-1] != source:
        prev = pred_array[path[-1]]
        if prev in visited:
            break
        visited.add(prev)
        path.append(prev)
    return list(reversed(path))

true_pred = predecessors_by_step[-1]
pred_pred = p_pred.argmax(dim=1).tolist()

print(f"\nShortest path prediction vs true (from source={source}):")
correct = 0
total = 0
for node in range(graph.num_nodes):
    if node == source:
        continue
    true_path = reconstruct_path(node, true_pred, source)
    pred_path = reconstruct_path(node, pred_pred, source)
    true_str = ' -> '.join(str(n) for n in true_path)
    pred_str = ' -> '.join(str(n) for n in pred_path)
    match = pred_path == true_path
    correct += match
    total += 1
    print(f"Node {node}: {pred_str} (expected: {true_str})")

print(f"\n{correct}/{total} shortest paths predicted correctly")

Step 1
Prediction: [-0.34, 1.26, 4.05, 1.31, 4.2, 0.58]
Target:     [0.0, 1.54, 4.0, 1.28, 4.0, 0.4]
Step 2
Prediction: [-0.6, 1.21, 2.22, 1.06, 2.21, 0.47]
Target:     [0.0, 1.54, 2.34, 1.28, 2.6, 0.4]

Shortest path prediction vs true (from source=0):
Node 1: 0 -> 1 (expected: 0 -> 1)
Node 2: 0 -> 3 -> 2 (expected: 0 -> 3 -> 2)
Node 3: 0 -> 3 (expected: 0 -> 3)
Node 4: 0 -> 1 -> 4 (expected: 0 -> 1 -> 4)
Node 5: 0 -> 5 (expected: 0 -> 5)

5/5 shortest paths predicted correctly
